# Toxic Comments Classification - SVM

#### Author: SAFAA ENNACIRI - ENNS11538307                                 
#### Research Director: M. HAKIM LOUNIS

Dataset: Toxic Comment Classification dataset is a dataset of comments from Wikipedia’s talk page edits.  It is avaialble at Kaggle (https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge/data). 


The comments are divided in two classes: toxic and normal.

# 1. Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle, re
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import LinearSVC
from calendar import month_name

# 2. Data Preparation 

In [19]:
# Load training data
df = pd.read_csv('../../Data/train_set2.csv')
df.head()

,Unnamed: 0,tweet,tox
0,15632,RT @FunSizedYogi: @TheBlackVoice well how else...,1
1,9609,Funny thing is....it's not just the people doi...,0
2,20769,"RT @winkSOSA: ""@AintShitSweet__: ""@Rakwon_OGOD...",1
3,3737,@Jbrendaro30 @ZGabrail @ramsin1995 @GabeEli8 @...,1
4,21026,S/o that real bitch,1


In [11]:
# remove the 10 most frequent (but keep the first one!)

dellist = pd.Series(' '.join(df['tweet']).split()).value_counts()[1:10]
print("Davidson\n")
print(dellist)

Davidson

RT       5994
bitch    5259
I        5199
the      5181
to       4216
you      3881
and      3000
my       2528
that     2507
dtype: int64


In [20]:
dellist_ = pd.Series(' '.join(df['tweet']).split()).value_counts(ascending=True)[1:10]
print("Davidson\n")
print(dellist_)

Davidson

@nikkiconner13             1
leader.                    1
Heru                       1
February                   1
@wwwobert_:                1
#Honda200                  1
Gotti                      1
Spittin                    1
http://t.co/8oxlQvQBrC"    1
dtype: int64


In [11]:
def c_text(comment): 
    comment = comment.lower() # Normalize to lowercase 

    if u'â' in comment:
        print (comment)
        print("--------------------------------------------")
    return comment

In [12]:
# clean and normalize comments
df['tweet'] = df['tweet'].map(lambda word:c_text(word))

In [3]:
for x in range(40):
    print(df['tweet'][x])

Just straight like that. No chaser. And the bitch was a SOLID FIVE ON HER WEDDING DAY MY NIGGA.
@blowme you mean trash.
"WHAAAAAAT! LMAOOO!!! @Cromotoasted: @creepgoddess that's why I luv u fat bitches cuz y'all got some big ole titties and phat booties"
RT @Tylar____: Man this hoe Heaven stay taking all day
Col. Sanders, Part 2: "...and those sons of bitches&#8212; they dragged it out and extended it and watered it down that I&#8217;m so goddamn mad!"
Lol this freshman said why dress up as a baby that's so retarded, I said well I mean u act like a baby so might as well dress like one
I drive by and them bitches waive but girl I can't stop
I used to picture him coming up the wadi to FOB Sharana with a barrel of brandy for me @iiarwatch
RT @RayIopez: "make me ur wcw"

"how many girls do u talk to"

"am I ur side hoe"

"tweet about me so I know it's real" http://t.co/LB274Gh&#8230;
I've got saltine crackers and red wine here. Is it possible to DIY a communion? I, uh, haven't been to chur

In [239]:
print(df['comment_text'][3])

15:19, 13 November 2014


In [240]:
print(df['comment_text'][14])

You can't delete shit from my talk page you faggot, get that shit out of here


In [241]:
print(df['comment_text'][11])

Is it gay if you bang an animal of the same secks?


In [242]:
print(df['comment_text'][5])

The second member of his surname is not spelled right... It starts with an F 

You know, Carter-F*ck


In [243]:
print(df['comment_text'][4])

if you have something to say to me say it now


# 3. Preprocessing

In [26]:
stop_words = set(stopwords.words('english'))
import contractions
from nltk.tokenize import word_tokenize
def clean_text(comment): 
   
    comment = re.sub(r'[^a-zA-Z]', ' ',comment) 
    #comment = re.sub('[\\r\\t\\n]', ' ', comment)
    #comment = re.sub(' +',' ', comment)
    comment = re.sub("@\S+", " ", comment) # Remove usernames
    comment = re.sub("https*\S+", " ", comment) # Remove URL
    #comment = re.sub("\\<([^/> ]+)", " ", comment) # Remove HTML tags
    #comment = comment.replace("#", "") #remode hashtag
    #comment = re.sub(r'[^a-zA-Z]', ' ',comment) 
    #comment = re.sub('[\\r\\t\\n]', ' ', comment)
   # comment = re.sub(' +',' ', comment)
    #comment = contractions.fix(comment)
    #months = {m.lower() for m in month_name[1:]}  # create a set of month names
    #for word in comment.split():
    #if word.lower() in months:
        #comment = comment.replace(word, "") 
   
  #  word_tokens = word_tokenize(comment)
    # converts the words in word_tokens to lower case and then checks whether 
    #they are present in stop_words or not
    #filtered_sentence = [w for w in word_tokens if not w.lower() in stop_words]
    #with no lower case conversion
    filtered_sentence = []

   # for w in word_tokens:
    #    if w not in stop_words:
     #       filtered_sentence.append(w)
  
    #sentence = (" ").join(filtered_sentence)
    
    return comment

In [27]:
# clean and normalize comments
df['tweet'] = df['tweet'].map(lambda word:clean_text(word))

In [25]:
# remove the 10 most frequent (but keep the first one!)

dellist = pd.Series(' '.join(df['tweet']).split()).value_counts(ascending=True)[1:10]
print("Davidson\n")
print(dellist)

Davidson

REALLY?           1
21,               1
"entertaining"    1
rant              1
They've           1
"lol"             1
gay&#8221;or      1
stroked           1
mandate           1
dtype: int64


In [28]:
dellist_ = pd.Series(' '.join(df['tweet']).split()).value_counts(ascending=True)[1:10]
print("Davidson\n")
print(dellist_)

Davidson

theeeese       1
concluded      1
remarks        1
restricton     1
eric           1
bellybutton    1
Chilli         1
Boom           1
crissi         1
dtype: int64


In [2]:
import emot
emot_obj = emot.core.emot() 
text = "I love python ☮ 🙂 ❤ :-) :-( :-)))" 
t ="def hgt wwq"
emot_obj.emoji(text) 


{'value': [], 'location': [], 'mean': [], 'flag': False}

In [41]:
 from emot.emo_unicode import EMOTICONS_EMO # For EMOTICONS

In [35]:
import pickle, re

# load file  
with open('../../Data/Emoji_Dict.p', 'rb') as fp:
    Emoji_Dict = pickle.load(fp)
Emoji_Dict = {v: k for k, v in Emoji_Dict.items()}

#remove empjis
def remove_emojis(text):
    for emoj in Emoji_Dict:
        text = re.sub(r'('+emoj+')', " ", text)
    return text

In [49]:
text1 = "I want some chinamen hella bad I wish this coronavirus shit would die down a little 😂😩 damn"
remove_emojis(text1)

'I want some chinamen hella bad I wish this coronavirus shit would die down a little    damn'

In [47]:
# load file
with open('../../Data/Emoticon_Dict.p', 'rb') as fl:
    Emoticon_Dict = pickle.load(fl)

# remove emoticons
def remove_emoticons(text):
    emoticon_pattern = re.compile(u'(' + u'|'.join(k for k in Emoticon_Dict) + u')')
    return emoticon_pattern.sub(r'', text)

remove_emoticons("Good Morning :-)")

'Good Morning '

In [48]:
text2 = "I won 🥇 in 🏏:-)"
remove_emoticons(text2)

'I won 🥇 in 🏏'

In [233]:
# clean and normalize comments
df['comment_text'] = df['comment_text'].map(lambda word:clean_text(word))

In [235]:
print(df['comment_text'][11])

['Is', 'it', 'gay', 'if', 'you', 'bang', 'an', 'animal', 'of', 'the', 'same', 'secks', '?']


In [98]:
print(df['comment_text'][5])

The second member of his surname is not spelled right... It starts with an F   You know, Carter-F*ck


In [179]:
print(df['comment_text'][3])

15:19, 13 November 2014


In [169]:
print(df['comment_text'][17])

hahaha, fuck you esa, you worthless pieces of shit.


In [151]:
stop_words = set(stopwords.words('english'))
print(stop_words)

{'most', 'from', 'that', 'those', 'on', 'yourself', 'ourselves', 'them', 'both', 'had', 'at', 'any', 'in', 'with', 'why', 'such', "haven't", 'have', 'the', 'be', 'when', 'more', 'can', 'won', 'a', 'whom', 'ma', 'into', 'over', 'your', 'about', 't', 'aren', 'between', 'needn', 'haven', 'as', 'further', 'ain', "needn't", "mightn't", "shouldn't", 'which', 'while', 'to', 'again', 'once', 'this', 'yours', "doesn't", 'being', 'off', 'themselves', 'i', 'been', 'has', "wouldn't", 'same', 'shouldn', 'out', "she's", 'our', 're', "should've", 'until', 'how', 'some', 'herself', 'shan', 'do', 'doesn', 'few', 'of', 'by', 'too', 'myself', 'these', 'itself', 'its', 'him', 'no', 'didn', 'other', 'hers', 'now', 'own', 'mightn', "shan't", "you've", 'd', 'an', 'don', "mustn't", "couldn't", "you'll", "weren't", 'hasn', 'it', 's', "you'd", 'ours', 'y', 'not', 'we', 'her', 'but', 'their', 'up', 'all', 'here', 'during', "won't", 'my', 'there', "hasn't", "it's", 'where', 'only', 'than', 'who', 'or', "hadn't", 

In [152]:
print(df['comment_text'][12])

"The ""jobber"" had a name. Rory Fox to be exact. Good to see laziness still exists. Poor Nici.. what a a prick.. 

"


In [153]:
example_sent = (df['comment_text'][4])
from nltk.tokenize import word_tokenize
word_tokens = word_tokenize(example_sent)
# converts the words in word_tokens to lower case and then checks whether 
#they are present in stop_words or not
filtered_sentence = [w for w in word_tokens if not w.lower() in stop_words]
#with no lower case conversion
filtered_sentence = []
  
for w in word_tokens:
    if w not in stop_words:
        filtered_sentence.append(w)
  
print(word_tokens)
print(filtered_sentence)

['if', 'you', 'have', 'something', 'to', 'say', 'to', 'me', 'say', 'it', 'now']
['something', 'say', 'say']


In [154]:
sentence = (" ").join(filtered_sentence)
print(sentence)

something say say


In [50]:
from string import punctuation
print(punctuation)


!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [32]:
w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer = nltk.stem.WordNetLemmatizer()

def lemmatization(text):
    """This function converts word to their root words 
       without explicitely cut down as done in stemming.
    
    arguments:
         input_text: "text" of type "String".
         
    return:
        value: Text having root words only, no tense form, no plural forms
        
    Example: 
    Input : text reduced 
    Output :  text reduce
    
   """
    # Converting words to their root forms
    lemma = [lemmatizer.lemmatize(w,'v') for w in w_tokenizer.tokenize(text)]
    return lemma

In [47]:
s =  (lemmatization("beautifulness"))

In [48]:
print(s)

['beautifulness']


In [34]:
sentence = (" ").join(s)
print(sentence)

You people love to step on other people, don't you?


In [2]:
import nltk
from nltk.stem import PorterStemmer
w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
word_stemmer = PorterStemmer()
def stemer(text):
      # Converting words to their root forms
    st = [word_stemmer.stem(w,'v') for w in w_tokenizer.tokenize(text)]
    return st
s = stemer('HE vandalized MINE using a sockpuppet accusation template.  *I* merely returned the favor.  Now blow it out your eye wall! ')
sentence = (" ").join(s)
print(sentence)

he vandal mine use a sockpuppet accus template. *i* mere return the favor. now blow it out your eye wall!


In [51]:
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
wordnet_map = {"N":wordnet.NOUN, "V":wordnet.VERB, "J":wordnet.ADJ, "R":wordnet.ADV} # Pos tag, used Noun, Verb, Adjective and Adverb
# Function for lemmatization using POS tag
def lemmatize_words(text):
    pos_tagged_text = nltk.pos_tag(text.split())
    return " ".join([lemmatizer.lemmatize(word, wordnet_map.get(pos[0], wordnet.NOUN)) for word, pos in pos_tagged_text])
# Passing the function to 'text_rare' and store in 'text_lemma'
#df["text_lemma"] = df["text_rare"].apply(lemmatize_words)

In [67]:
s =  (lemmatization("aardwolves"))
sentence = (" ").join(s)
print(sentence)

aardwolves


In [72]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()
print(wnl.lemmatize('dogs'))
#dog
print(wnl.lemmatize('churches'))
#church
print(wnl.lemmatize('aardwolves'))
#aardwolf
print(wnl.lemmatize('abaci'))
#abacus
print(wnl.lemmatize('hardrock'))
#hardrock
print(wnl.lemmatize('doing'))

dog
church
aardwolf
abacus
hardrock
doing


In [135]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

text = "using returned sweetly careful tweets i am"
wordnet = WordNetLemmatizer()
tokenizer = word_tokenize(text)
s = []

for token in tokenizer:
    s.append(wordnet.lemmatize(token))
sentence = (" ").join(s)
print(sentence)

using returned sweetly careful tweet i am


In [134]:
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag

text1 = "using returned sweetly careful tweets I am"
wordnet = WordNetLemmatizer()
def lemmatize_token1 (text):
    s = []

    for token,tag in pos_tag(word_tokenize(text)):
        pos=tag[0].lower()
        
        if pos not in ['a', 'r', 'v', 's']:
            pos='n'
    
        s.append(wordnet.lemmatize(token,pos))
    
    sentence = (" ").join(s)
    return sentence
print(lemmatize_token1(text1))

use return sweetly careful tweet I be


In [129]:
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag

wordnet = WordNetLemmatizer()
def lemmatize_token (text):
    lemma = []

    for token,tag in pos_tag(word_tokenize(text)):
        pos = tag[0].lower()        
        if pos not in ['v', 'r', 'a', 's']:
            pos = 'n'    
        lemma.append(wordnet.lemmatize(token,pos))    
    sentence = (" ").join(s)
    return sentence

In [130]:
print(lemmatize_token(text1))

using returned sweetly aardwolf


In [183]:
comment = "13 November 2014 march 2000"
months = {m.lower() for m in month_name[1:]}  # create a set of month names
for word in comment.split():
    if word.lower() in months:
        comment = comment.replace(word, "")


In [184]:
comment

'13  2014  2000'

In [5]:
print("Ensemble de données Davidson")
df.isnull().sum()

Ensemble de données Davidson


Unnamed: 0    0
tweet         0
tox           0
dtype: int64